<a href="https://colab.research.google.com/github/14marcos1/pibicjr/blob/main/exames001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install playwright
!playwright install chromium
!playwright install-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 14.8 MB/s eta 0:00:00
(node:7670) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 344.7s167.3 MiB [] 0% 59.5s167.3 MiB [] 0% 34.2s167.3 MiB [] 0% 24.7s167.3 MiB [] 0% 23.9s167.3 MiB [] 0% 18.7s167.3 MiB [] 0% 12.8s167.3 MiB [] 1% 10.6s167.3 MiB [] 1% 9.2s167.3 MiB [] 2% 8.2s167.3 MiB [] 2% 7.9s167.3 MiB [] 2% 8.2s167.3 MiB [] 3% 7.2s167.3 MiB [] 3% 6.8s167.3 MiB [] 4% 6.0s167.3 MiB [] 4% 5.6s167.3 MiB [] 5% 5.3s167.3 MiB [] 5% 5.0s167.3 MiB [] 6% 4.8s167.3 MiB [] 6% 4.6s167.3 MiB [] 7% 4.4s167.3 MiB [] 7% 4.3s167.3 MiB [] 8% 4.2s167.3 MiB [] 9% 4.1s167.3 MiB [] 9% 4.0s167.3 MiB [] 10% 3.9s167.3 MiB [] 10% 3.7s167.3 MiB [] 11% 3.6s167.3 MiB [] 12% 3.5s167.3 MiB [] 13% 3.4s167.3 M

In [2]:
import asyncio
from playwright.async_api import async_playwright
import os

async def rodar_scraping():
    URL = "http://tabnet.datasus.gov.br/cgi/deftohtm.exe?sia/cnv/qamg.def"

    async with async_playwright() as p:
        # No Colab, headless deve ser True e precisamos de alguns argumentos extras
        navegador = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        contexto = await navegador.new_context(accept_downloads=True)
        pagina = await contexto.new_page()

        print("Acessando a URL...")
        await pagina.goto(URL)
        await asyncio.sleep(2)

        # Seleções iniciais (L, C e I)
        await pagina.locator('xpath=//*[@id="L"]/option[15]').click()
        await pagina.locator('xpath=//*[@id="C"]/option[11]').click()
        await pagina.locator('xpath=//*[@id="I"]/option[1]').click()

        # Configurações de filtros adicionais
        await pagina.locator('xpath=//*[@id="fig8"]').click()
        await pagina.locator('xpath=//*[@id="S8"]/option[3]').click()
        await pagina.locator('xpath=//*[@id="fig9"]').click()
        await pagina.locator('xpath=//*[@id="S8"]/option[3]').click()

        meses = []
        # Loop para clicar nos meses e abrir novas abas
        print("Iniciando a abertura das abas de períodos...")
        for idx in range(11, 14):  # Ajuste conforme necessário
            opcao = pagina.locator(f'xpath=//*[@id="A"]/option[{idx}]')
            texto_mes = (await opcao.inner_text()).strip()
            meses.append(texto_mes)

            await opcao.click()

            # Clica no botão 'Mostra' e espera abrir a nova aba
            async with contexto.expect_page() as new_page_info:
                await pagina.locator('xpath=/html/body/div[1]/div/center/div/form/div[4]/div[2]/div[2]/input[1]').click()

            print(f"Aba para {texto_mes} aberta.")

        # Pausa para garantir carregamento das abas
        await asyncio.sleep(5)

        # As páginas de resultado são todas exceto a primeira (índice 0)
        paginas_resultado = contexto.pages[1:]

        for mes, page_res in zip(meses, paginas_resultado):
            await page_res.bring_to_front()

            # Nome do arquivo
            mes_safe = mes.replace("/", "-").replace(" ", "_")
            nome_arquivo = f"datasus_2026_{mes_safe}.csv"

            try:
                # Tenta baixar o CSV
                async with page_res.expect_download() as download_info:
                    await page_res.locator('xpath=/html/body/div/div/div[3]/table/tbody/tr/td[1]/a').click()

                download = await download_info.value
                await download.save_as(nome_arquivo)
                print(f"✅ Arquivo salvo: {nome_arquivo}")
            except Exception as e:
                print(f"❌ Erro ao baixar {mes}: {e}")

        await navegador.close()
        print("\nProcesso concluído! Verifique a pasta lateral do Colab.")

# Comando para rodar no ambiente Jupyter/Colab
await rodar_scraping()

Acessando a URL...
Iniciando a abertura das abas de períodos...
Aba para Abr/2025 aberta.
Aba para Mar/2025 aberta.
Aba para Fev/2025 aberta.
✅ Arquivo salvo: datasus_2026_Abr-2025.csv
✅ Arquivo salvo: datasus_2026_Mar-2025.csv
✅ Arquivo salvo: datasus_2026_Fev-2025.csv

Processo concluído! Verifique a pasta lateral do Colab.
